# Step 7: Model Training
Train the Isolation Forest; set contamination and the Low / Medium / High cut-offs from managers' review capacity (MF4); build the simple policy-rules baseline (MF4).

**Input:** `data/processed/features.csv`  
**Output:** `models/isolation_forest.joblib`

## Part A: First model
Train Isolation Forest on the 8 features from Step 6, turn its output into a 0–1 score (higher = more unusual),
apply the review-capacity cut-offs (high = top 1%, medium = next 2%), and check what gets flagged.
_Code for this step comes next._

In [1]:
# 7.1 Load the features from Step 5 and the final list from Step 6.
# We read the list from models/feature_list.json instead of re-typing it, so training, evaluation and
# the ML service always use exactly the same features in the same order.
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.read_csv("../data/processed/features.csv", parse_dates=["expense_date"])
with open("../models/feature_list.json", encoding="utf-8") as fh:
    FEATURES = json.load(fh)["features"]

X = df[FEATURES]
print("Rows:", len(df), "| features:", len(FEATURES))
print(FEATURES)

Rows: 209012 | features: 8
['log_amount', 'amount_vs_employee_category', 'category_share_for_employee', 'same_category_day_count', 'duplicate_count', 'is_weekend', 'is_round_amount', 'other_category_day_count']


In [2]:
# 7.2 Isolation Forest: builds many random trees. Each tree keeps splitting the data on a random feature
# at a random value until every request sits alone. Unusual requests get isolated in FEW splits
# (they're far from the crowd); normal ones need many.
#   n_estimators=200   → 200 trees; more trees = a more stable score
#   max_samples="auto" → each tree sees a random 256 requests (fast, and it's what the original paper uses)
#   contamination="auto" → we DON'T let the model choose the cut-off; we set it ourselves from review capacity (7.4)
#   random_state=42    → the same trees on every run (repeatable results)
#   n_jobs=-1          → use all CPU cores
model = IsolationForest(n_estimators=200, max_samples="auto", contamination="auto",
                        random_state=42, n_jobs=-1)
model.fit(X)
print("Trees:", len(model.estimators_), "| requests per tree:", model.max_samples_)

Trees: 200 | requests per tree: 256


In [3]:
# 7.3 score_samples() returns NEGATIVE numbers where lower = more unusual. Putting a minus in front gives
# the anomaly score from the Isolation Forest paper: between 0 and 1, HIGHER = MORE UNUSUAL,
# which is exactly the "score" field in the contract (B4). Around 0.5 or less = normal; near 1 = very unusual.
df["score"] = -model.score_samples(X)
print(df["score"].describe().round(3))

count    209012.000
mean          0.408
std           0.065
min           0.344
25%           0.362
50%           0.383
75%           0.433
max           0.748
Name: score, dtype: float64


In [4]:
# 7.4 Review capacity (MF4): high = the top 1% most unusual, medium = the next 2% (so 3% flagged in total).
# quantile(0.99) = the score that 99% of requests are below → everything at or above it is the top 1%.
# These two numbers are the placeholders until the team confirms; change them here.
HIGH_SHARE = 0.01
MEDIUM_SHARE = 0.03          # high + medium together

high_from = df["score"].quantile(1 - HIGH_SHARE)
medium_from = df["score"].quantile(1 - MEDIUM_SHARE)

df["risk_level"] = np.select([df["score"] >= high_from, df["score"] >= medium_from],
                             ["high", "medium"], default="low")
print(f"medium from {medium_from:.3f} | high from {high_from:.3f}")
print(df["risk_level"].value_counts())

medium from 0.569 | high from 0.632
risk_level
low       202741
medium      4179
high        2092
Name: count, dtype: int64


In [5]:
# 7.5 Compare the typical (median) high-risk request with the typical request overall,
# then look at the 5 most unusual requests and the employee-days with the most high-risk requests.
compare = pd.DataFrame({
    "all requests": df[FEATURES].median(),
    "high risk":    df.loc[df["risk_level"] == "high", FEATURES].median(),
})
print(compare.round(2))

cols = ["requester_id", "expense_date", "category", "amount_sar", "amount_vs_employee_category",
        "same_category_day_count", "duplicate_count", "is_weekend", "score"]
top = df.sort_values("score", ascending=False)[cols].head(5)
top["score"] = top["score"].round(2)
print(top.to_string(index=False))

print(df[df["risk_level"] == "high"].groupby(["requester_id", "expense_date"]).size()
        .sort_values(ascending=False).head(3))

                             all requests  high risk
log_amount                           5.87       5.81
amount_vs_employee_category         -0.01      -0.69
category_share_for_employee          0.59       0.49
same_category_day_count              1.00      21.00
duplicate_count                      1.00      11.00
is_weekend                           0.00       0.00
is_round_amount                      0.00       0.00
other_category_day_count             0.00       0.00
 requester_id expense_date category  amount_sar  amount_vs_employee_category  same_category_day_count  duplicate_count  is_weekend  score
         1373   2025-06-27   Travel        45.0                    -6.450274                       26               21           1   0.75
         1373   2025-06-27   Travel        45.0                    -6.450274                       26               21           1   0.75
         1373   2025-06-27   Travel        45.0                    -6.450274                       26        

In [6]:
# 7.6 Three checks: what share of each kind of request got flagged (medium or high)?
#  - amounts far ABOVE the person's normal → we WANT these flagged
#  - amounts far BELOW the person's normal → cheaper than usual isn't a risk; we DON'T want these
#  - busy days (5+ requests in one category) → common in this card data (see Step 6)
flagged = df["risk_level"] != "low"
checks = {
    "Amount 5+ spreads ABOVE the person's normal": df["amount_vs_employee_category"] >= 5,
    "Amount 3+ spreads BELOW the person's normal": df["amount_vs_employee_category"] <= -3,
    "5+ requests in one category on one day":      df["same_category_day_count"] >= 5,
}
for name, mask in checks.items():
    print(f"{name:45} {mask.sum():6} requests | flagged: {flagged[mask].mean():5.1%}")
print("Share of all flagged that are busy days:", f"{checks['5+ requests in one category on one day'][flagged].mean():.0%}")

Amount 5+ spreads ABOVE the person's normal      370 requests | flagged: 17.3%
Amount 3+ spreads BELOW the person's normal     1813 requests | flagged: 15.8%
5+ requests in one category on one day         25565 requests | flagged: 18.0%
Share of all flagged that are busy days: 74%


## Part B: Final model (hybrid design)
Part A showed that the same-day counts take over the model (74% of flags were busy days), that it flags cheaper-than-usual
amounts, and that it can't learn splits from card data (in PCard, several purchases a day is normal).
**Decision: hybrid.** The model learns "unusual for this person" from 5 features; the same-day patterns
(split, duplicate, many requests) become clear rules in Part C. Amounts are made one-sided before the model,
so only *higher than normal* counts.

In [7]:
# 7.7 The hybrid design: 5 features go into the model, 3 are used by the rules (Part C).
# The 3 rule features are still computed exactly as in Step 5; they're just not inside the forest.
MODEL_FEATURES = ["log_amount", "amount_vs_employee_category", "category_share_for_employee",
                  "is_weekend", "is_round_amount"]
RULE_FEATURES = ["same_category_day_count", "duplicate_count", "other_category_day_count"]

# Together they must be exactly the 8 features chosen in Step 6
assert set(MODEL_FEATURES + RULE_FEATURES) == set(FEATURES)

In [8]:
# 7.8 Isolation Forest isolates BOTH ends of every feature, so a much CHEAPER request than usual looked
# just as "unusual" as a much more expensive one (Part A: 15.8% of very cheap ones flagged).
# Fix: before the model, cut off the side we don't care about:
#   amount_vs_employee_category: everything below 0 (= at or under the person's normal) becomes 0
#   log_amount: everything below the median (~352 SAR) becomes the median ("small is small")
# LOG_AMOUNT_FLOOR is a fixed number from the training data. The ML service must use the SAME number,
# so it's saved with the model in Part D.
# Reasons still use the real (uncut) values; this only changes what the model sees.
LOG_AMOUNT_FLOOR = round(float(df["log_amount"].median()), 4)
print("log_amount floor:", LOG_AMOUNT_FLOOR, "=", round(np.expm1(LOG_AMOUNT_FLOOR)), "SAR")

def model_input(data):
    X = data[MODEL_FEATURES].copy()
    X["log_amount"] = X["log_amount"].clip(lower=LOG_AMOUNT_FLOOR)
    X["amount_vs_employee_category"] = X["amount_vs_employee_category"].clip(lower=0)
    return X

X_model = model_input(df)
print(X_model.describe().T[["min", "50%", "max"]].round(2))

log_amount floor: 5.8661 = 352 SAR
                              min   50%    max
log_amount                   5.87  5.87  15.22
amount_vs_employee_category  0.00  0.00  10.00
category_share_for_employee  0.00  0.59   1.00
is_weekend                   0.00  0.00   1.00
is_round_amount              0.00  0.00   1.00


In [9]:
# 7.9 The same settings as 7.2, now on the 5 one-sided model features.
# Then the same review-capacity cut-offs as 7.4 (HIGH_SHARE = 1%, MEDIUM_SHARE = 3%).
model = IsolationForest(n_estimators=200, max_samples="auto", contamination="auto",
                        random_state=42, n_jobs=-1)
model.fit(X_model)

df["score"] = -model.score_samples(X_model)
high_from = df["score"].quantile(1 - HIGH_SHARE)
medium_from = df["score"].quantile(1 - MEDIUM_SHARE)
df["risk_level"] = np.select([df["score"] >= high_from, df["score"] >= medium_from],
                             ["high", "medium"], default="low")
print(f"medium from {medium_from:.3f} | high from {high_from:.3f}")
print(df["risk_level"].value_counts())

medium from 0.627 | high from 0.664
risk_level
low       202741
medium      4179
high        2092
Name: count, dtype: int64


In [10]:
# 7.10 The Part A checks again, plus "first time in this category". high = only the top 1%.
flagged = df["risk_level"] != "low"
high = df["risk_level"] == "high"
checks = {
    "Amount 5+ spreads ABOVE the person's normal": df["amount_vs_employee_category"] >= 5,
    "Amount 3+ spreads BELOW the person's normal": df["amount_vs_employee_category"] <= -3,
    "First request ever in this category":         (df["category_share_for_employee"] == 0)
                                                    & df["baseline_level"].isin(["empcat", "emp"]),
    "5+ requests in one category on one day":      df["same_category_day_count"] >= 5,
}
for name, mask in checks.items():
    print(f"{name:45} {mask.sum():6} | flagged: {flagged[mask].mean():6.1%} | high: {high[mask].mean():6.1%}")

# What a typical high-risk request looks like now, and the 5 most unusual
compare = pd.DataFrame({"all requests": df[MODEL_FEATURES].median(),
                        "high risk":    df.loc[high, MODEL_FEATURES].median()})
print(compare.round(2))
print("Weekend: all", f"{df['is_weekend'].mean():.1%}", "| high", f"{df.loc[high, 'is_weekend'].mean():.1%}")
print("Round:   all", f"{df['is_round_amount'].mean():.1%}", "| high", f"{df.loc[high, 'is_round_amount'].mean():.1%}")

cols = ["requester_id", "category", "amount_sar", "amount_vs_employee_category",
        "category_share_for_employee", "is_weekend", "score"]
print(df.sort_values("score", ascending=False)[cols].head(5).round(3).to_string(index=False))

Amount 5+ spreads ABOVE the person's normal      370 | flagged:  99.7% | high:  51.9%
Amount 3+ spreads BELOW the person's normal     1813 | flagged:   0.2% | high:   0.1%
First request ever in this category             4925 | flagged:  10.5% | high:   4.9%
5+ requests in one category on one day         25565 | flagged:   2.4% | high:   0.5%
                             all requests  high risk
log_amount                           5.87       8.76
amount_vs_employee_category         -0.01       2.19
category_share_for_employee          0.59       0.35
is_weekend                           0.00       0.00
is_round_amount                      0.00       0.00
Weekend: all 9.2% | high 46.3%
Round:   all 0.7% | high 37.7%
 requester_id                category  amount_sar  amount_vs_employee_category  category_share_for_employee  is_weekend  score
         3440 Equipment & Maintenance    183750.0                        3.564                        0.486           0  0.786
         1197         

In [11]:
# 7.11 In the app, the model is trained on the past and then scores NEW requests. Check that the cut-offs
# still flag about 1% / 3% on months the model has never seen:
# train on everything before April 2026, set the cut-offs there, then score April-June 2026.
cut = pd.Timestamp("2026-04-01")
past, future = df[df["expense_date"] < cut], df[df["expense_date"] >= cut]

m_past = IsolationForest(n_estimators=200, max_samples="auto", contamination="auto",
                         random_state=42, n_jobs=-1).fit(model_input(past))
s_past = -m_past.score_samples(model_input(past))
s_future = -m_past.score_samples(model_input(future))
hi_p, med_p = np.quantile(s_past, [1 - HIGH_SHARE, 1 - MEDIUM_SHARE])

print(f"Trained on {len(past)} requests before {cut.date()}, checked on {len(future)} after")
print(f"Flagged in the past months: high {np.mean(s_past >= hi_p):.2%} | high+medium {np.mean(s_past >= med_p):.2%}")
print(f"Flagged in the new months:  high {np.mean(s_future >= hi_p):.2%} | high+medium {np.mean(s_future >= med_p):.2%}")

Trained on 157466 requests before 2026-04-01, checked on 51546 after
Flagged in the past months: high 1.00% | high+medium 3.00%
Flagged in the new months:  high 1.02% | high+medium 3.25%


In [12]:
# 7.12 The real check for MF4: how many flags does each manager get per week?
# Only the real year (Jun 2025 - Jun 2026; the few earlier rows are late postings).
# W-SAT = weeks that end on Saturday, so a week is Sunday-Saturday like the Saudi week.
real_year = df[(df["expense_date"] >= "2025-06-29") & (df["expense_date"] <= "2026-06-27")].copy()
real_year["week"] = real_year["expense_date"].dt.to_period("W-SAT")

weekly = (real_year.groupby(["department_id", "week"])
            .agg(requests=("expense_id", "size"),
                 high=("risk_level", lambda s: (s == "high").sum()),
                 medium=("risk_level", lambda s: (s == "medium").sum()))
            .groupby("department_id").mean().round(1)
            .sort_values("requests"))
print(weekly.iloc[[0, 1, 9, 10, -2, -1]])     # 2 smallest, 2 in the middle, 2 biggest

               requests  high  medium
department_id                        
17                 28.0   0.6     0.7
20                 41.3   0.6     1.7
8                  83.7   0.7     1.8
10                 84.8   0.9     2.5
2                 668.0   5.7    12.7
1                1127.4  11.1    21.2


## Part C: Rules for same-day patterns
Splits, duplicates and many requests in one day are Masroof policy patterns that card data treats as normal,
so they're clear rules instead of model features. Each rule is strict (card data has many multi-purchase days).
Final risk level: high = the model's top 1% (careful check); medium = the model's next 1% OR any rule (quick look).

In [13]:
# 7.13 The rules. Each returns True/False per request. All settings are at the top so they're easy to change.
# CATEGORY_LIMITS = the same per-request limits as the backend's EXPENSE_LIMITS (contract E5).
CATEGORY_LIMITS = {"Food": 2000, "Travel": 10000, "Transportation": 5000,
                   "Office Supplies": 3000, "IT & Telecom": 7500, "Equipment & Maintenance": 7500}

# Split: 2-5 requests in one category on one day, EACH under the limit, TOGETHER over it,
# and each part at least a quarter of the limit (tiny add-ons aren't an attempt to dodge a limit).
# More than 5 parts = a bulk-buying day, not a split.
SPLIT_MIN_PART = 0.25
SPLIT_MAX_PARTS = 5

# Duplicate: the same amount in the same category on the same day, 2-3 copies, at least 1,000 SAR.
# 4+ identical copies = bulk buying (e.g. 21 x 45 SAR travel fees), not an accidental double submit.
DUPLICATE_MIN_AMOUNT = 1000
DUPLICATE_MAX_COPIES = 3

# Many requests: 8+ requests the same day in OTHER categories than this one.
MANY_OTHER_FROM = 8

def apply_rules(data):
    # For each employee + day + category: the total, the smallest and the largest amount
    same_day = data.groupby(["requester_id", "expense_date", "category"])["amount_sar"]
    total = same_day.transform("sum")
    smallest = same_day.transform("min")
    largest = same_day.transform("max")
    limit = data["category"].map(CATEGORY_LIMITS)
    n = data["same_category_day_count"]

    split = ((n >= 2) & (n <= SPLIT_MAX_PARTS) & (largest <= limit) & (total > limit)
             & (smallest >= SPLIT_MIN_PART * limit))
    duplicate = (data["duplicate_count"].between(2, DUPLICATE_MAX_COPIES)
                 & (data["amount_sar"] >= DUPLICATE_MIN_AMOUNT))
    many = data["other_category_day_count"] >= MANY_OTHER_FROM

    # The column names are the reason codes from the contract, so a True becomes that reason
    return pd.DataFrame({"possible_split_request": split,
                         "duplicate_request": duplicate,
                         "many_requests_same_day": many}, index=data.index)

rules = apply_rules(df)
print(rules.mean().mul(100).round(2))
print("Any rule:", f"{rules.any(axis=1).mean():.2%}")

possible_split_request    0.68
duplicate_request         1.23
many_requests_same_day    0.38
dtype: float64
Any rule: 2.21%


In [14]:
# 7.14 Employee 1157 on 14 Jan 2026 (the split from Step 5): 3 Food requests, each under the 2,000 limit,
# 2,021.25 together, each part above 500 (a quarter of the limit) → all 3 should be marked.
ex = df[(df["requester_id"] == 1157) & (df["expense_date"] == "2026-01-14")]
print(pd.concat([ex[["expense_id", "category", "amount_sar"]], rules.loc[ex.index]], axis=1).to_string(index=False))

 expense_id category  amount_sar  possible_split_request  duplicate_request  many_requests_same_day
     231604     Food      544.61                    True              False                   False
     231605     Food      688.84                    True              False                   False
     231606     Food      787.80                    True              False                   False


In [15]:
# 7.15 Combine:
#   high   = the model's top 1% (HIGH_SHARE from 7.4): the careful-check budget, rules don't add to it
#   medium = the model's next 1% (its top 2% minus the high ones) OR any rule fired: a quick look
#   low    = everything else
# MODEL_FLAG_SHARE was 3% in Part A; it's 2% now to leave room for the rules.
MODEL_FLAG_SHARE = 0.02
medium_from = df["score"].quantile(1 - MODEL_FLAG_SHARE)

model_level = np.select([df["score"] >= high_from, df["score"] >= medium_from],
                        ["high", "medium"], default="low")
any_rule = rules.any(axis=1)
df["risk_level"] = np.where(model_level == "high", "high",
                    np.where((model_level == "medium") | any_rule, "medium", "low"))

print(f"model medium from {medium_from:.3f} | high from {high_from:.3f}")
print((df["risk_level"].value_counts(normalize=True) * 100).round(2))

# Do the model and the rules flag the SAME requests, or different ones?
model_flag = model_level != "low"
print(pd.Series({"model only": (model_flag & ~any_rule).mean(),
                 "rules only": (~model_flag & any_rule).mean(),
                 "both":       (model_flag & any_rule).mean()}).mul(100).round(2))

model medium from 0.641 | high from 0.664
risk_level
low       95.92
medium     3.08
high       1.00
Name: proportion, dtype: float64
model only    1.87
rules only    2.08
both          0.13
dtype: float64


In [16]:
# 7.16 The same weekly check as 7.12, now with the final risk level (model + rules).
real_year = df[(df["expense_date"] >= "2025-06-29") & (df["expense_date"] <= "2026-06-27")].copy()
real_year["week"] = real_year["expense_date"].dt.to_period("W-SAT")
weekly = (real_year.groupby(["department_id", "week"])
            .agg(requests=("expense_id", "size"),
                 high=("risk_level", lambda s: (s == "high").sum()),
                 medium=("risk_level", lambda s: (s == "medium").sum()))
            .groupby("department_id").mean().round(1)
            .sort_values("requests"))
print(weekly.iloc[[0, 1, 9, 10, -2, -1]])

               requests  high  medium
department_id                        
17                 28.0   0.6     2.3
20                 41.3   0.6     4.0
8                  83.7   0.7     1.8
10                 84.8   0.9     3.9
2                 668.0   5.7    17.9
1                1127.4  11.1    30.0


In [17]:
# 7.17 In Part A the model alone caught 0 of 100 planted splits. The same kind of split
# (3 Office Supplies requests on one day, each under 3,000, together 8,350) through the rules:
planted = pd.DataFrame({
    "requester_id": [9999] * 3,
    "expense_date": [pd.Timestamp("2026-11-18")] * 3,
    "category": ["Office Supplies"] * 3,
    "amount_sar": [2950.0, 2800.0, 2600.0],
    "same_category_day_count": [3] * 3,
    "duplicate_count": [1] * 3,
    "other_category_day_count": [0] * 3,
})
print(pd.concat([planted[["category", "amount_sar"]], apply_rules(planted)], axis=1).to_string(index=False))

       category  amount_sar  possible_split_request  duplicate_request  many_requests_same_day
Office Supplies      2950.0                    True              False                   False
Office Supplies      2800.0                    True              False                   False
Office Supplies      2600.0                    True              False                   False


## Part D: Save the model and its settings
`models/isolation_forest.joblib` = the trained model; `models/model_config.json` = every number chosen in this notebook
(cut-offs, amount floor, rule settings, limits). Step 8 and the ML service (Step 9) load both, so they use exactly this model.

In [18]:
# 7.18 joblib saves the trained Python object (all 200 trees) to a file; joblib.load() gives it back
# exactly as it is, so the service never has to retrain.
# MODEL_VERSION goes into every ML response ("model_version" in the contract) and replaces "stub-0.2".
# Bump it every time the model is retrained, so the backend knows to re-score (rescore_ai --all).
import os
import joblib
import sklearn

MODEL_VERSION = "iforest-0.1"
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/isolation_forest.joblib")
print("Model file:", round(os.path.getsize("../models/isolation_forest.joblib") / 1e6, 2), "MB")

Model file: 2.29 MB


In [19]:
# 7.19 Every number the service needs to give the SAME answer as this notebook:
#   model_features + model_input → what goes into the model, and the one-sided floors (7.8)
#   cut_offs → where medium and high start (7.9, 7.15)
#   rules + category_limits → the 3 rules (7.13)
#   feature_settings → MIN_HISTORY, MIN_STD, Z_CAP from notebook 04 (5.6), for features.py in Step 9
#   trained_on → what data it learned from; sklearn_version matters because a model saved with one
#   scikit-learn version may not load in a different one, so the service must use the same version
config = {
    "model_version": MODEL_VERSION,
    "trained_on": {
        "requests": int(len(df)),
        "from": str(df["expense_date"].min().date()),
        "to": str(df["expense_date"].max().date()),
        "sklearn_version": sklearn.__version__,
    },
    "model_features": MODEL_FEATURES,
    "rule_features": RULE_FEATURES,
    "model_input": {"log_amount_floor": LOG_AMOUNT_FLOOR, "amount_vs_employee_category_floor": 0},
    "cut_offs": {
        "high_from": float(high_from),
        "medium_from": float(medium_from),
        "high_share": HIGH_SHARE,
        "model_flag_share": MODEL_FLAG_SHARE,
    },
    "rules": {
        "split_min_part": SPLIT_MIN_PART,
        "split_max_parts": SPLIT_MAX_PARTS,
        "duplicate_min_amount": DUPLICATE_MIN_AMOUNT,
        "duplicate_max_copies": DUPLICATE_MAX_COPIES,
        "many_other_from": MANY_OTHER_FROM,
    },
    "category_limits": CATEGORY_LIMITS,
    "feature_settings": {"min_history": 5, "min_std": 0.1, "z_cap": 10},
}
with open("../models/model_config.json", "w", encoding="utf-8") as fh:
    json.dump(config, fh, indent=2)
print(json.dumps(config["cut_offs"], indent=2))

{
  "high_from": 0.6638820970206413,
  "medium_from": 0.6411856293776573,
  "high_share": 0.01,
  "model_flag_share": 0.02
}


In [20]:
# 7.20 Load both files the way the ML service will, score everything again, and check that we get
# exactly the same scores and risk levels as above. If this passes, the saved files ARE the model.
loaded = joblib.load("../models/isolation_forest.joblib")
with open("../models/model_config.json", encoding="utf-8") as fh:
    cfg = json.load(fh)

def risk_from(scores, any_rule, cfg):
    # The final rule from 7.15, using only the saved numbers
    c = cfg["cut_offs"]
    return np.where(scores >= c["high_from"], "high",
           np.where((scores >= c["medium_from"]) | any_rule, "medium", "low"))

check_scores = -loaded.score_samples(model_input(df))
print("Same scores:", np.allclose(check_scores, df["score"]))
print("Same risk levels:", (risk_from(check_scores, any_rule, cfg) == df["risk_level"]).all())
print(cfg["model_version"], "| trained on", cfg["trained_on"]["requests"], "requests")

Same scores: True
Same risk levels: True
iforest-0.1 | trained on 209012 requests


## Result
- **Hybrid design.** Model: Isolation Forest (200 trees, 256 requests each) on 5 "unusual for this person" features, with one-sided amounts
  (log_amount floor ≈ 352 SAR, amount_vs_employee_category floor 0). Rules: split, duplicate, many requests in one day.
- **Why hybrid (Part A):** with all 8 features, 74% of flags were busy days, cheaper-than-usual amounts got flagged,
  and 0 of 100 planted splits were caught (several purchases a day is normal on a company card).
- **Final model:** 99.7% of amounts 5+ spreads above the person's normal are flagged (52% high); cheaper ones 0.2%.
- **Risk levels:** high = model top 1% (careful check); medium = model next 1% or any rule (quick look). Total ≈ 4%.
- **Model and rules catch different requests:** model only 1.87%, rules only 2.08%, both 0.13%.
- **Stable on new months:** trained before April 2026 → 1.02% high / 3.25% flagged by the model on Apr–Jun.
- **Per manager per week:** biggest department 11 high + 30 medium; a typical one < 1 high + 2–4 medium.
- Saved: `models/isolation_forest.joblib` (2.3 MB), `models/model_config.json` (version `iforest-0.1`).